# Part A - Load, Inspect, and Clean

This notebook loads the raw COVID-19 dataset, inspects it, handles missing values and outliers, computes rolling means, and saves the cleaned CSV for use by the Flask app.

In [1]:
import pandas as pd

df = pd.read_csv('COVID_Country_Sample.csv', parse_dates=['date'])
print('Shape:', df.shape)
df.head(10)

Shape: (300, 8)


,date,country,new_cases,new_deaths,new_vaccinations,population,cases_per_million,vaccinations_per_hundred
0,2020-01-01,Canada,383,1,0.0,38000000,10.09,0.0
1,2020-02-01,Canada,582,2,0.0,38000000,15.32,0.0
2,2020-03-01,Canada,738,10,0.0,38000000,19.43,0.0
3,2020-04-01,Canada,692,8,0.0,38000000,18.24,0.0
4,2020-05-01,Canada,392,3,0.0,38000000,10.32,0.0
5,2020-06-01,Canada,171,2,0.0,38000000,4.53,0.0
6,2020-07-01,Canada,197,2,0.0,38000000,5.20,0.0
7,2020-08-01,Canada,223,3,0.0,38000000,5.89,0.0
8,2020-09-01,Canada,132,1,0.0,38000000,3.49,0.0
9,2020-10-01,Canada,247,1,0.0,38000000,6.51,0.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      300 non-null    datetime64[ns]
 1   country                   300 non-null    object        
 2   new_cases                 300 non-null    int64         
 3   new_deaths                300 non-null    int64         
 4   new_vaccinations          294 non-null    float64       
 5   population                300 non-null    int64         
 6   cases_per_million         300 non-null    float64       
 7   vaccinations_per_hundred  300 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int64(3), object(1)
memory usage: 18.9+ KB


## Missing Values

Check which columns have NaNs and how many.

In [3]:
df.isnull().sum()

date                        0
country                     0
new_cases                   0
new_deaths                  0
new_vaccinations            6
population                  0
cases_per_million           0
vaccinations_per_hundred    0
dtype: int64

All 6 NaNs are in `new_vaccinations`. These occur in early 2020 (pre-rollout) or isolated reporting gaps. Filling with 0 is semantically correct - no doses were administered in the pre-rollout period, and 0 is a conservative choice for later gaps.

In [4]:
df['new_vaccinations'] = df['new_vaccinations'].fillna(0)
print('NaNs after fill:', df.isnull().sum().sum())

NaNs after fill: 0


## Outlier Handling

Use the 3x IQR outer fence per (country, metric). Values above Q3 + 3*IQR are capped rather than dropped - this preserves the full timeline while preventing extreme spikes from collapsing the y-axis.

In [5]:
NUMERIC_COLS = ['new_cases', 'new_deaths', 'new_vaccinations', 'cases_per_million']

def cap_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    upper = q3 + 3 * (q3 - q1)
    return series.clip(upper=upper)

for col in NUMERIC_COLS:
    df[col] = df.groupby('country')[col].transform(cap_outliers)

df[NUMERIC_COLS].describe()

,new_cases,new_deaths,new_vaccinations,cases_per_million
count,300.000000,300.000000,3.000000e+02,300.000000
mean,436.577500,3.396667,1.121773e+08,4.619250
std,347.492287,3.302880,2.418356e+08,5.818417
min,40.000000,1.000000,0.000000e+00,0.040000
25%,217.750000,1.000000,9.361438e+05,0.687500
50%,355.500000,2.000000,2.233545e+07,2.065000
75%,533.250000,4.250000,8.405415e+07,6.540000
max,2172.250000,21.000000,1.443989e+09,38.845000


## Rolling Means

Compute a 3-month rolling mean per country for each numeric metric. This is used as an optional overlay in the Flask app to smooth short-term noise.

In [6]:
df = df.sort_values(['country', 'date'])

for col in NUMERIC_COLS:
    df[f'{col}_rolling'] = (
        df.groupby('country')[col]
        .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

df[['date', 'country', 'new_cases', 'new_cases_rolling']].head(10)

,date,country,new_cases,new_cases_rolling
0,2020-01-01,Canada,383.0,383.000000
1,2020-02-01,Canada,582.0,482.500000
2,2020-03-01,Canada,738.0,567.666667
3,2020-04-01,Canada,692.0,670.666667
4,2020-05-01,Canada,392.0,607.333333
5,2020-06-01,Canada,171.0,418.333333
6,2020-07-01,Canada,197.0,253.333333
7,2020-08-01,Canada,223.0,197.000000
8,2020-09-01,Canada,132.0,184.000000
9,2020-10-01,Canada,247.0,200.666667


## Save Cleaned CSV

Write the cleaned dataframe to disk. The Flask app loads this file directly, so cleaning only needs to run once here in the notebook.

In [7]:
df.to_csv('COVID_Country_Sample_cleaned.csv', index=False)
print('Saved:', df.shape)
df.head()

Saved: (300, 12)


,date,country,new_cases,new_deaths,new_vaccinations,population,cases_per_million,vaccinations_per_hundred,new_cases_rolling,new_deaths_rolling,new_vaccinations_rolling,cases_per_million_rolling
0,2020-01-01,Canada,383.0,1,0.0,38000000,10.09,0.0,383.000000,1.000000,0.0,10.090000
1,2020-02-01,Canada,582.0,2,0.0,38000000,15.32,0.0,482.500000,1.500000,0.0,12.705000
2,2020-03-01,Canada,738.0,10,0.0,38000000,19.43,0.0,567.666667,4.333333,0.0,14.946667
3,2020-04-01,Canada,692.0,8,0.0,38000000,18.24,0.0,670.666667,6.666667,0.0,17.663333
4,2020-05-01,Canada,392.0,3,0.0,38000000,10.32,0.0,607.333333,7.000000,0.0,15.996667
